# RadiologyAI — Linha de base honesta no Google Colab

**O que este notebook produz:** *um* número de desempenho real, medido, reproduzível,
com intervalo de confiança — e o artefato `metrics.json` que o sustenta.

É o marco da Fase 1 do [ROADMAP](https://github.com/drguilhermecapel/radiologyai/blob/main/ROADMAP.md):
substituir todas as métricas fabricadas do repositório v1 por uma medição verificável.

---

## A armadilha que este notebook evita

O caminho óbvio seria rodar `densenet121-res224-all` no NIH ChestX-ray14. **Não faça isso.**

Os pesos `-all` foram treinados em NIH, PadChest, CheXpert, MIMIC-CXR, Google, OpenI e RSNA —
o próprio nome do arquivo publicado é `nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-...`.
Avaliá-los no NIH é **in-distribution**: produziria um número inflado que parece medição mas não é.
Seria uma nova fabricação, só que mais sutil que a do v1.

Este notebook usa **`densenet121-res224-pc`** — treinado só em PadChest (Hospital San Juan,
Alicante, Espanha) — avaliado no **split oficial de teste do NIH** (EUA, 25.596 imagens,
disjunto por paciente). País, equipamento, população e pipeline de rotulagem diferentes:
**validação externa genuína, sem credenciamento, custo zero.**

O código detecta vazamento automaticamente e recusa alegar validação externa quando não há.

---

## Antes de começar

1. `Ambiente de execução -> Alterar o tipo de ambiente de execução -> GPU T4`
2. A GPU não é obrigatória (funciona em CPU), mas reduz a inferência de ~2h para ~12min.
3. O disco do Colab é **efêmero**. O dataset (42 GB) é baixado a cada sessão; só os
   artefatos (poucos KB) são salvos no seu Drive.


## 1. Ambiente


In [ ]:
!nvidia-smi 2>/dev/null || echo 'Sem GPU — vai funcionar em CPU, só mais devagar'
!df -h /content | tail -1
import sys; print('Python', sys.version.split()[0])


O Colab roda Python 3.12; o pacote exige 3.11. Instalamos sem a checagem de versão
(`--no-deps` no próprio pacote) e as dependências explicitamente. É uma concessão
consciente ao ambiente do Colab: o pino `>=3.11,<3.12` continua valendo para CI e produção.


In [ ]:
REPO = 'https://github.com/drguilhermecapel/radiologyai.git'
BRANCH = 'claude/roadmap-interpretacao-radiologica-vh15ek'

!git clone --depth 1 --branch {BRANCH} {REPO} /content/radiologyai
%cd /content/radiologyai
!git log --oneline -1


In [ ]:
# Dependências. torch já vem no Colab.
!pip install -q pydicom==2.4.4 'pydantic>=2.7' 'pydantic-settings>=2.3' \
    'typer>=0.12' torchxrayvision scikit-image pillow
!pip install -q --no-deps -e .

import sys; sys.path.insert(0, '/content/radiologyai/src')
import radiologyai; print('radiologyai', radiologyai.__version__)


## 2. Onde salvar os artefatos

O dataset **não** vai para o Drive: 42 GB não cabem na conta gratuita, e não
precisam — os pixels são reproduzíveis a partir da fonte. O que persiste é o
`metrics.json` e o manifest, que somam poucos megabytes e são o registro de
reprodutibilidade.

Montar o Drive abre uma janela de autorização do Google. **Se você quer deixar
rodando sem supervisão, pule esta célula** — os artefatos ficam no disco do
Colab e a última seção oferece o download direto. O risco de pular: se a sessão
cair, você perde o resultado junto com o disco.


In [ ]:
from pathlib import Path

# Deixe False para não montar o Drive (evita a janela de autorização).
USAR_DRIVE = True

ARTIFACTS = Path('/content/artifacts/eval')

if USAR_DRIVE:
    try:
        from google.colab import drive

        drive.mount('/content/drive')
        ARTIFACTS = Path('/content/drive/MyDrive/radiologyai/artifacts/eval')
    except Exception as erro:
        # Falhar aqui não deve derrubar a execução: os artefatos continuam
        # sendo gravados, só que no disco efêmero do Colab.
        print(f'Drive não montado ({erro}). Usando o disco local do Colab.')
        print('O resultado será perdido se a sessão cair — baixe-o ao final.')

ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('artefatos em:', ARTIFACTS)


## 3. Baixar o NIH ChestX-ray14

112.120 radiografias frontais de 30.805 pacientes, do NIH Clinical Center.
Livre, sem registro e sem credenciamento.

### Sobre a credencial do Kaggle

**Você não precisa do arquivo `kaggle.json`.** A CLI do Kaggle lê as variáveis
`KAGGLE_USERNAME` e `KAGGLE_KEY`, e a célula abaixo as pega do **Colab Secrets**
automaticamente.

Se ainda não configurou: clique no **ícone de chave** na barra lateral esquerda e
crie dois secrets, ambos com *Notebook access* ligado para este notebook:

| Nome do secret | Valor |
|---|---|
| `KAGGLE_USERNAME` | seu nome de usuário do Kaggle |
| `KAGGLE_KEY` | o token de https://www.kaggle.com/settings → *Create New Token* |

O token vem dentro do `kaggle.json` que o site baixa: abra o arquivo e copie o
valor do campo `key`. Não é preciso subir o arquivo para lugar nenhum.

### As duas rotas

| | Kaggle | NIH oficial |
|---|---|---|
| Credencial | Colab Secrets | **nenhuma** |
| Fonte | `nih-chest-xrays/data` | Box (imagens) + Hugging Face (metadados) |
| Velocidade | geralmente mais rápida | moderada |
| Disco | ~45 GB (dataset completo) | ~10 GB com `--test-only` |

A célula usa **`ROTA = 'auto'`**: pega o Kaggle se houver credencial no Secrets,
e cai para a fonte oficial do NIH se não houver. Nos dois casos funciona.

Os links das duas rotas foram verificados. Os links diretos do Box para os
*metadados* mudaram e retornam 404 — por isso o script os busca do espelho do
Hugging Face, e confere cabeçalho e contagem de linhas antes de prosseguir:
um download truncado é detectado agora, não depois de horas de inferência.


In [ ]:
DATA = Path('/content/nih')
DATA.mkdir(parents=True, exist_ok=True)

# ROTA = 'auto' usa o Kaggle se houver credencial no Colab Secrets, e cai para a
# fonte oficial do NIH caso contrário. Force com 'kaggle' ou 'nih' se quiser.
ROTA = 'auto'


def credencial_kaggle() -> bool:
    """Lê a credencial do Kaggle do Colab Secrets (ícone de chave, à esquerda).

    A CLI do Kaggle aceita as variáveis de ambiente KAGGLE_USERNAME e KAGGLE_KEY.
    **Nenhum arquivo kaggle.json é necessário.**
    """
    import os

    try:
        from google.colab import userdata
    except ImportError:
        return False

    # Nomes usuais, caso você tenha salvo com outra capitalização.
    for nome_usuario, nome_chave in [('KAGGLE_USERNAME', 'KAGGLE_KEY'),
                                     ('kaggle_username', 'kaggle_key'),
                                     ('KaggleUsername', 'KaggleKey')]:
        try:
            usuario = userdata.get(nome_usuario)
            chave = userdata.get(nome_chave)
        except Exception:
            continue
        if usuario and chave:
            os.environ['KAGGLE_USERNAME'] = usuario.strip()
            os.environ['KAGGLE_KEY'] = chave.strip()
            print(f"Credencial do Kaggle lida do Secrets (usuário: {usuario.strip()})")
            return True
    return False


usar_kaggle = ROTA == 'kaggle' or (ROTA == 'auto' and credencial_kaggle())

if usar_kaggle:
    if ROTA == 'kaggle' and not credencial_kaggle():
        raise SystemExit(
            'Sem credencial no Secrets. Crie KAGGLE_USERNAME e KAGGLE_KEY no ícone '
            'de chave (com acesso a este notebook), ou use ROTA = "nih".'
        )
    print('Rota: Kaggle (~45 GB)')
    !pip install -q kaggle
    !kaggle datasets download -d nih-chest-xrays/data -p {DATA} --unzip
else:
    print('Rota: NIH oficial, sem credencial.')
    print('Se você tem conta no Kaggle, criar KAGGLE_USERNAME e KAGGLE_KEY no')
    print('Colab Secrets (ícone de chave) costuma ser bem mais rápido.')
    # --test-only extrai apenas as 25.596 imagens do split de teste: economiza
    # ~30 GB de disco. O download continua sendo dos 12 tarballs, porque as
    # imagens de teste estão espalhadas entre eles.
    !python scripts/fetch_nih_cxr14.py --out {DATA} --test-only


In [ ]:
# Os arquivos oficiais precisam existir e conferir. O split de teste do NIH já
# é disjunto por paciente — é por isso que usamos o oficial em vez do nosso.
ESPERADO = {'Data_Entry_2017_v2020.csv': 112120, 'test_list.txt': 25596}

ok = True
for nome, n_esperado in ESPERADO.items():
    caminho = DATA / nome
    if not caminho.exists():
        print(f'FALTA  {nome}')
        ok = False
        continue
    linhas = caminho.read_text().splitlines()
    n = len(linhas) - 1 if nome.endswith('.csv') else len([x for x in linhas if x.strip()])
    marca = 'OK ' if n == n_esperado else '!! '
    print(f'{marca}{nome}: {n} (esperado {n_esperado})')
    ok = ok and n == n_esperado

n_imagens = sum(1 for _ in DATA.rglob('*.png'))
print(f'\nimagens em disco: {n_imagens}  (o teste usa 25.596)')
!du -sh {DATA}

if not ok:
    raise SystemExit('Metadados incompletos. Uma avaliação sobre isso não significa nada.')


## 4. Construir o manifest

O manifest é o registro de reprodutibilidade: quais imagens, de qual paciente, com quais
rótulos. Ele é commitado no repositório; os pixels nunca são.


In [ ]:
!python -m radiologyai.cli.main manifest {DATA} \
    --out /content/radiologyai/datasets/manifests/nih_cxr14_test.csv \
    --split test


## 5. Rodar a avaliação

Este é o único caminho pelo qual um número de desempenho pode entrar no repositório.
O artefato grava `git_sha`, `weights_sha256`, `manifest_sha256`, `seed` e as versões
das bibliotecas — o conjunto necessário para demonstrar que a execução é repetível
(IEC 62304 §5.7).

Tempo estimado: ~12 min em T4, ~2 h em CPU.


In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('dispositivo:', DEVICE)

!python -m radiologyai.cli.main evaluate \
    --card xrv-densenet121-pc \
    --manifest /content/radiologyai/datasets/manifests/nih_cxr14_test.csv \
    --data-root {DATA} \
    --out {ARTIFACTS} \
    --device {DEVICE} \
    --batch-size 64 \
    --bootstrap 2000 \
    --seed 20260101


## 6. Ler o resultado

**Expectativa: AUROC macro entre 0,72 e 0,82.** Cardiomegalia, derrame e enfisema devem
ficar altos (0,85–0,90); pneumonia, infiltrado e nódulo baixos (0,65–0,73).

O README do v1 alegava 0,94. **Publicar 0,78 com intervalo de confiança é o objetivo
inteiro deste marco.** Um número menor e verdadeiro vale mais que um número maior e inventado.


In [ ]:
import json
run_dir = sorted(ARTIFACTS.glob('xrv-densenet121-pc__*'))[-1]
m = json.loads((run_dir / 'metrics.json').read_text())

print('AUROC macro:', m['macro_auroc'])
print('rótulos avaliados:', m['n_labels_evaluated'])
print('externo ao treino:', m['dataset']['external_to_training_data'],
      f"({m['dataset']['leakage_status']})")
print('imagens/pacientes:', m['dataset']['n_images'], '/', m['dataset']['n_patients'])
print()
for name, e in sorted(m['per_label'].items(), key=lambda kv: -kv[1]['auroc']):
    lo, hi = e['auroc_ci95']
    print(f"  {name:<26} {e['auroc']:.3f}  IC95 [{lo:.3f}, {hi:.3f}]  n+={e['n_pos']:>5}")


In [ ]:
# Subgrupos — requisito de equidade. Uma lacuna grande é achado a reportar,
# não a esconder. A diferença PA vs AP costuma ser visível e é ela própria um achado.
for group, values in m['subgroups'].items():
    print(f'\n{group}:')
    for k, v in values.items():
        print(f"  {k:<12} n={v['n']:>6}  AUROC macro={v['macro_auroc']}")


In [ ]:
# O que NÃO foi avaliado — declarado, nunca descartado em silêncio
for x in m['not_evaluated']:
    print(f"  {x['label']:<28} {x['reason']}")
print()
print('LIMITAÇÕES DECLARADAS:')
for l in m['limitations']:
    print(' *', l)


## 7. Levar o resultado de volta ao repositório

O artefato é o que autoriza qualquer alegação. Sem ele, o verificador de CI
`scripts/check_honesty.py` quebra o build de qualquer documento que cite um número.


In [ ]:
import shutil
dest = Path('/content/radiologyai/artifacts/eval') / run_dir.name
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_dir, dest, dirs_exist_ok=True)

!cd /content/radiologyai && python scripts/check_honesty.py
print('\nArquivos a commitar:')
!cd /content/radiologyai && git status --short artifacts/ datasets/


In [ ]:
# Baixe o artefato para o seu computador e commite no repositório.
# Faça isto SEMPRE que não estiver usando o Drive.
import shutil

shutil.make_archive('/content/baseline', 'zip', run_dir)
print('pronto:', run_dir.name)

try:
    from google.colab import files

    files.download('/content/baseline.zip')
except Exception as erro:
    print(f'Download automático falhou ({erro}).')
    print('Baixe /content/baseline.zip pelo painel de arquivos, à esquerda.')


---

## O que este resultado é — e o que não é

**É:** uma medição retrospectiva de desempenho de algoritmo isolado, num conjunto de
teste externo ao treino do modelo, com intervalo de confiança e análise de subgrupos.

**Não é:** validação clínica. Não é evidência de utilidade clínica. Não é medição do
modelo *do produto* — este é um modelo de terceiros usado como referência. Os escores
não são calibrados e não representam probabilidade de doença.

Os rótulos do NIH são **minerados por NLP dos laudos**, não adjudicados por radiologista.
A validação contra rótulos adjudicados vem na Fase 3, com o **VinDr-CXR** (18.000 exames,
conjunto de teste lido por 5 radiologistas) — que exige credenciamento PhysioNet.

**Comece o credenciamento agora:** curso CITI "Data or Specimens Only Research", gratuito,
~6 h, e a aprovação leva de 2 a 6 semanas. Como médico com CRM ativo você é elegível.
Este notebook foi desenhado para não depender de nada credenciado, justamente para que
o credenciamento corra em paralelo e nunca fique no caminho crítico.
